# LangChain Sales Assistant

A beginner-friendly project to learn LangChain concepts while building a practical sales assistant.

## What You'll Learn
- **Prompt Templates**: Reusable prompt structures
- **LLM Chains**: Connecting multiple LLM calls
- **Output Parsing**: Structuring LLM responses
- **Memory**: Maintaining conversation context
- **Tools & Functions**: Making LLMs take actions

## Project Goals
Build a sales assistant that:
1. Analyzes and scores leads
2. Researches company information
3. Generates personalized sales pitches
4. Drafts outreach emails

## Setup & Installation

Run these commands in your terminal first:
```bash
pip install -r ../requirements.txt
ollama pull mistral  # if not already done
```

Then make sure Ollama is running:
```bash
ollama serve
```

In [5]:
# Import required libraries
import json
import os
from typing import Optional, List, Dict
from pprint import pprint

from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM

print("✅ All imports successful!")

✅ All imports successful!


## Step 1: Initialize Ollama LLM

This connects to your local Ollama server running Mistral model.

In [6]:
# Initialize the Ollama LLM
# Make sure Ollama is running: ollama serve
llm = OllamaLLM(model="mistral", temperature=0.7)

# Test the connection
test_response = llm.invoke("Say 'Hello from Mistral' in one sentence.")
print("Test response from Mistral:")
print(test_response)
print("\n✅ Connection to Ollama successful!")

Test response from Mistral:
 "Greetings from Mistral: Bonjour du Mistral!"

✅ Connection to Ollama successful!


In [3]:
llm.invoke("Say exactly: Hello from Mistral. Do not use any other words.")

' Hello from Mistral.'

## Step 2: Load Sample Leads

Load the sample leads data to work with.

In [8]:
# Load sample leads
with open('../data/sample_leads.json', 'r') as f:
    leads = json.load(f)

print(f"Loaded {len(leads)} sample leads")
print("\nFirst lead:")
pprint(leads[0])

Loaded 2 sample leads

First lead:
{'budget': '$10000',
 'company': 'TechStartup Inc',
 'company_size': '20-50 employees',
 'contact_name': 'John Smith',
 'current_tools': 'Spreadsheets, basic CRM',
 'decision_timeline': 'Q4 2024',
 'id': 'lead_001',
 'industry': 'SaaS',
 'pain_points': 'Manual lead tracking, poor sales forecasting'}


---

# PART 1: Lead Analysis & Qualification

## Concept: Prompt Templates + LCEL (LangChain Expression Language)

**Prompt Template**: A template with placeholders that gets filled with variables

**LCEL (|)**: The pipe operator chains components together. `prompt | llm` means: pass prompt output to LLM input. This is the modern, recommended way to build chains in LangChain!

In [15]:
# Create a Prompt Template for lead analysis

lead_analysis_prompt = PromptTemplate(
    input_variables=["company", "industry", "pain_points", "budget"],
    template="""Act as an expert veteran Sales Analyst responsible for qualifying leads.
Your job is to filter out low-quality prospects and pass only high-potential leads 
to our sales team, saving them time on unqualified prospects.

Analyze this sales lead:

Company: {company}
Industry: {industry}
Pain Points: {pain_points}
Budget: {budget}

Use these criteria for evaluation:
- Budget should indicate purchasing power (higher budget = better)
- Pain points should align with common industry challenges
- Company size and industry matter for fit

Provide a JSON response with:
- lead_quality_score (0-100): Higher if high budget + clear pain points
- fit_assessment (high/medium/low): Based on industry fit
- key_opportunities (list of 3-5 strings): Specific selling points
- recommendation (buy/nurture/pass): buy = immediate action, pass = not ready
- confidence_level (0-100): How confident in your assessment
- reasoning (2-3 sentences explaining your assessment: why this score, how industry fits, budget implications)"""
)

print("📋 Prompt Template created!")
print("Template variables:", lead_analysis_prompt.input_variables)

📋 Prompt Template created!
Template variables: ['budget', 'company', 'industry', 'pain_points']


In [16]:
# Create a chain using LCEL (the pipe operator)
# prompt | llm means: chain the prompt template with the LLM

lead_analysis_chain = lead_analysis_prompt | llm

print("⛓️ Chain created using LCEL!")
print("Chain formula: prompt_template | llm")
print(f"  - Prompt template with variables: {lead_analysis_prompt.input_variables}")
print(f"  - LLM: {llm.model}")

⛓️ Chain created using LCEL!
Chain formula: prompt_template | llm
  - Prompt template with variables: ['budget', 'company', 'industry', 'pain_points']
  - LLM: mistral


In [17]:
# Test the chain with a lead

lead = leads[0]

# With LCEL, invoke returns the result directly (not wrapped in a dict)
result = lead_analysis_chain.invoke({
    "company": lead["company"],
    "industry": lead["industry"],
    "pain_points": lead["pain_points"],
    "budget": lead["budget"]
})

print("🎯 Lead Analysis Result:")
print(result)  # result is now a string directly!

🎯 Lead Analysis Result:
 {
  "lead_quality_score": 80,
  "fit_assessment": "high",
  "key_opportunities": [
    "Automation of manual lead tracking",
    "Improved sales forecasting",
    "Integration with existing SaaS ecosystem",
    "Cost savings through efficient lead management"
  ],
  "recommendation": "buy",
  "confidence_level": 90,
  "reasoning": "The lead shows a clear budget indication and specific pain points that align with common challenges in the SaaS industry. The budget of $10,000 is reasonable for the solutions being offered, and the company's industry makes for a good fit. The lead's readiness to address these pain points suggests they are actively looking to invest in a solution, making them a high-quality prospect."
}


## Output Parsing: Converting Text to Structured Data

The LLM returns text, but we want structured data (JSON). Let's parse it!

In [18]:
# Create an improved prompt that forces JSON output

lead_analysis_prompt_v2 = PromptTemplate(
    input_variables=["company", "industry", "pain_points", "budget"],
    template="""Analyze this sales lead and respond ONLY with valid JSON (no other text):
    
    Company: {company}
    Industry: {industry}
    Pain Points: {pain_points}
    Budget: {budget}
    
    Respond with ONLY this JSON structure:
    {{
      "lead_quality_score": <0-100>,
      "fit_assessment": "high or medium or low",
      "key_opportunities": ["opp1", "opp2", "opp3"],
      "recommendation": "buy or nurture or pass",
      "confidence_level": <0-100>
    }}
    
    Only output the JSON, nothing else."""
)

# Create improved chain using LCEL
lead_analysis_chain_v2 = lead_analysis_prompt_v2 | llm

print("✅ Improved chain with better JSON formatting created!")

✅ Improved chain with better JSON formatting created!


In [19]:
# Test the improved chain

result = lead_analysis_chain_v2.invoke({
    "company": lead["company"],
    "industry": lead["industry"],
    "pain_points": lead["pain_points"],
    "budget": lead["budget"]
})

print("🎯 Lead Analysis Result (Raw):")
print(result)

# Try to parse as JSON
try:
    analysis_json = json.loads(result)
    print("\n📊 Parsed JSON:")
    pprint(analysis_json)
except json.JSONDecodeError as e:
    print(f"\n⚠️ Could not parse as JSON: {e}")
    print("Raw response:")
    print(result)

🎯 Lead Analysis Result (Raw):
 {
  "lead_quality_score": "80",
  "fit_assessment": "medium",
  "key_opportunities": ["Improved Lead Management Solutions", "Advanced Sales Forecasting Tools", "Integration with Existing Systems"],
  "recommendation": "nurture",
  "confidence_level": "75"
}

📊 Parsed JSON:
{'confidence_level': '75',
 'fit_assessment': 'medium',
 'key_opportunities': ['Improved Lead Management Solutions',
                       'Advanced Sales Forecasting Tools',
                       'Integration with Existing Systems'],
 'lead_quality_score': '80',
 'recommendation': 'nurture'}


In [20]:
# Create a reusable lead qualification function

def qualify_lead(lead_data: dict) -> dict:
    """
    Analyze and qualify a lead using LangChain LCEL.
    
    Args:
        lead_data: Dictionary with lead information
    
    Returns:
        Dictionary with lead analysis results
    """
    # LCEL chains return results directly (as strings)
    result = lead_analysis_chain_v2.invoke({
        "company": lead_data.get("company", ""),
        "industry": lead_data.get("industry", ""),
        "pain_points": lead_data.get("pain_points", ""),
        "budget": lead_data.get("budget", "")
    })
    
    try:
        analysis = json.loads(result)
        return {
            "lead_id": lead_data.get("id"),
            "company": lead_data.get("company"),
            "analysis": analysis
        }
    except json.JSONDecodeError:
        return {
            "lead_id": lead_data.get("id"),
            "company": lead_data.get("company"),
            "error": "Failed to parse LLM response"
        }

print("✅ Reusable lead qualification function created!")

✅ Reusable lead qualification function created!


In [21]:
# Test the function with all sample leads

print("🔍 Analyzing all sample leads...\n")

for lead in leads:
    result = qualify_lead(lead)
    print(f"Lead: {result['company']}")
    if "analysis" in result:
        print(f"  Quality Score: {result['analysis']['lead_quality_score']}/100")
        print(f"  Recommendation: {result['analysis']['recommendation']}")
    else:
        print(f"  Error: {result.get('error')}")
    print()

🔍 Analyzing all sample leads...

Lead: TechStartup Inc
  Quality Score: 75/100
  Recommendation: nurture

Lead: Retail Solutions Co
  Quality Score: 75/100
  Recommendation: nurture



---

# PART 2: Sales Research Module

## Concept: Building Chains with LCEL

The pipe operator (`|`) makes it easy to chain multiple components together. We can use the same pattern to build different chains for different tasks.

In [22]:
# Create a prompt for researching company information

research_prompt = PromptTemplate(
    input_variables=["company", "industry", "pain_points"],
    template="""Act as an expert veteran industry analyst. Based on the company and industry information, provide research insights:
    
    Company: {company}
    Industry: {industry}
    Pain Points: {pain_points}
    
    Provide 2-3 sentences on:
    - Industry trends including revenue growth
    - Typical solutions used
    - Competitive landscape
    - Buying trends"""
)

# Create the research chain using LCEL
research_chain = research_prompt | llm

print("✅ Research chain created using LCEL!")

✅ Research chain created using LCEL!


In [23]:
# Test research on a lead

lead = leads[0]
research_result = research_chain.invoke({
    "company": lead["company"],
    "industry": lead["industry"],
    "pain_points": lead["pain_points"]
})

print("🔬 Research Results:")
print(research_result)

🔬 Research Results:
 1. Industry Trends: The SaaS industry has been experiencing significant growth, with a projected compound annual growth rate (CAGR) of 12.4% from 2020 to 2025, according to MarketsandMarkets. Key trends include the increasing adoption of cloud-based solutions, integration of AI and machine learning, and a shift towards subscription-based business models.

2. Typical Solutions Used: Common solutions in the SaaS industry include customer relationship management (CRM) systems, marketing automation tools, and sales forecasting software. These tools help businesses manage their leads, automate marketing processes, and gain insights into sales trends, thereby addressing the pain points of manual lead tracking and poor sales forecasting.

3. Competitive Landscape: The SaaS market is highly competitive, with a large number of established players and new entrants. Key players include Salesforce, Microsoft Dynamics, HubSpot, and Zoho, among others. These companies offer comp

---

# PART 3: Pitch Generation

## Concept: Using Previous Results to Generate New Content

Combine lead analysis + research to create personalized pitches.

In [26]:
# Create a prompt for generating personalized sales pitches

pitch_prompt = PromptTemplate(
    input_variables=["company", "contact_name", "industry", "pain_points"],
    template="""Act as a seasoned Sales Executive with 25 years of winning business and earning sales excellence awards. Create a personalized sales pitch for this prospect:
    
    Contact: {contact_name}
    Company: {company}
    Industry: {industry}
    Pain Points: {pain_points}
    
    Write a compelling 2-3 paragraph elevator pitch that:
    1. Shows you understand their industry and problems and opportunities.
    2. Shows you understand their pain points
    3. Highlights specific benefits for their industry
    4. Includes a clear call-to-action
    
    Make it professional but conversational."""
)

# Create the pitch chain using LCEL
pitch_chain = pitch_prompt | llm

print("✅ Pitch generation chain created using LCEL!")

✅ Pitch generation chain created using LCEL!


In [27]:
# Generate a pitch for the first lead

lead = leads[0]

# Generate the pitch - LCEL returns the result directly
pitch_result = pitch_chain.invoke({
    "company": lead["company"],
    "contact_name": lead["contact_name"],
    "industry": lead["industry"],
    "pain_points": lead["pain_points"]
})

print(f"💼 Sales Pitch for {lead['contact_name']} ({lead['company']}):")
print("\n" + pitch_result)

💼 Sales Pitch for John Smith (TechStartup Inc):

 Dear John,

I hope this message finds you well. As a fellow innovator in the dynamic SaaS industry, I wanted to reach out and discuss an opportunity that could significantly streamline your operations and drive growth for TechStartup Inc.

I understand that in the fast-paced world of SaaS, manual lead tracking and inaccurate sales forecasting can be major hurdles. These issues not only consume valuable time and resources but also limit your ability to capitalize on opportunities. I've seen many companies, including TechStartup Inc, struggle with these pain points.

However, I'm excited to introduce you to our innovative solution designed specifically for businesses like yours. Our platform automates lead tracking, ensuring no potential opportunity slips through the cracks. By automating this process, your sales team can focus on what they do best - closing deals.

Moreover, our advanced forecasting tools provide accurate and reliable sa

---

# PART 4: Complete Sales Workflow

## Putting It All Together

Create a complete pipeline that analyzes, researches, and generates pitches for any lead.

In [28]:
def process_lead_complete(lead_data: dict) -> dict:
    """
    Complete sales workflow using LCEL:
    1. Analyze & qualify the lead
    2. Generate a personalized pitch
    """
    
    print(f"\n🚀 Processing lead: {lead_data['company']}")
    print("=" * 60)
    
    # Step 1: Qualify the lead
    print("\n📋 Step 1: Analyzing lead...")
    analysis = qualify_lead(lead_data)
    
    if "analysis" in analysis:
        print(f"✅ Quality Score: {analysis['analysis']['lead_quality_score']}/100")
        print(f"   Recommendation: {analysis['analysis']['recommendation']}")
    else:
        print("❌ Analysis failed")
        return analysis
    
    # Step 2: Generate pitch
    print("\n💼 Step 2: Generating personalized pitch...")
    pitch_result = pitch_chain.invoke({
        "company": lead_data["company"],
        "contact_name": lead_data["contact_name"],
        "industry": lead_data["industry"],
        "pain_points": lead_data["pain_points"]
    })
    print("✅ Pitch generated")
    
    # Return complete results
    return {
        "lead_id": lead_data["id"],
        "company": lead_data["company"],
        "analysis": analysis["analysis"] if "analysis" in analysis else None,
        "pitch": pitch_result
    }

print("✅ Complete workflow function created!")

✅ Complete workflow function created!


In [29]:
# Run the complete workflow on all leads

results = []

for lead in leads:
    result = process_lead_complete(lead)
    results.append(result)

print("\n\n🎉 Complete workflow finished!")
print(f"Processed {len(results)} leads")


🚀 Processing lead: TechStartup Inc

📋 Step 1: Analyzing lead...
✅ Quality Score: 70/100
   Recommendation: nurture

💼 Step 2: Generating personalized pitch...
✅ Pitch generated

🚀 Processing lead: Retail Solutions Co

📋 Step 1: Analyzing lead...
✅ Quality Score: 85/100
   Recommendation: buy

💼 Step 2: Generating personalized pitch...
✅ Pitch generated


🎉 Complete workflow finished!
Processed 2 leads


In [30]:
# Display results for review

for i, result in enumerate(results, 1):
    print(f"\n{'='*70}")
    print(f"LEAD #{i}: {result['company']}")
    print(f"{'='*70}")
    
    if result['analysis']:
        print(f"\n📊 Analysis:")
        print(f"  Quality Score: {result['analysis']['lead_quality_score']}/100")
        print(f"  Fit: {result['analysis']['fit_assessment']}")
        print(f"  Recommendation: {result['analysis']['recommendation']}")
    
    print(f"\n💼 Sales Pitch:")
    print(result['pitch'])


LEAD #1: TechStartup Inc

📊 Analysis:
  Quality Score: 70/100
  Fit: medium
  Recommendation: nurture

💼 Sales Pitch:
 Dear John,

I hope this message finds you well. I'm reaching out as the Head of Sales at XYZ Solutions, a leading provider of sales enablement tools, and I believe I have an offering that could significantly streamline your operations at TechStartup Inc.

In the dynamic and competitive SaaS industry like yours, managing leads manually and forecasting sales can be a daunting task. I understand that these challenges are not just time-consuming but also hinder your ability to seize opportunities and drive growth.

That's where XYZ Solutions comes in. Our cutting-edge platform is designed specifically for SaaS companies like yours, offering a robust solution to automate lead tracking and improve sales forecasting. With XYZ, you can focus on what truly matters - closing deals and growing your business.

I'd be delighted to arrange a demonstration of our platform at your ea

---

## Congratulations! 🎉

You've learned the modern fundamentals of LangChain:
- **Prompt Templates**: Reusable prompt structures for different tasks
- **LCEL (|)**: The pipe operator for building clean, composable chains
- **Output Parsing**: Converting LLM text responses to structured data (JSON)
- **Chaining**: Combining multiple operations together

This approach using LCEL is the recommended, modern way to build LLM applications. It's cleaner, more composable, and easier to reason about than the older LLMChain approach.

You now have a real, working sales assistant powered by LLMs!